# Data aggregation for 2026 matches

In [28]:
import joblib
import pandas as pd

## Load data & model

In [29]:
df_team_stats = pd.read_csv("model/team_stats.csv")
df_matches = pd.read_csv("predictions/predictions_2026_knockout_matches.csv")

clf = joblib.load("model/worldcup_model.pkl")
feature_cols = joblib.load("model/feature_columns.pkl")

### function to join team features to matches

In [30]:
def join_team_features_to_matches(matches):
    matches = matches.merge(
        df_team_stats,
        left_on="team1",
        right_on="team",
        how="left"
    )

    matches = matches.rename(
        columns={col: f"{col}_team1" for col in df_team_stats.columns if col != "team"}
    )
    matches = matches.drop(columns=["team"], errors="ignore")


    matches = matches.merge(
        df_team_stats,
        left_on="team2",
        right_on="team",
        how="left"
    )

    matches = matches.rename(
        columns={col: f"{col}_team2" for col in df_team_stats.columns if col != "team"}
    )
    matches = matches.drop(columns=["team"], errors="ignore")


    global_means = df_team_stats.mean(numeric_only=True)
    for col in global_means.index:
        matches[f"{col}_team1"] = matches[f"{col}_team1"].fillna(global_means[col])
        matches[f"{col}_team2"] = matches[f"{col}_team2"].fillna(global_means[col])

    matches["goals_diff"] = matches["total_goals_team1"] - matches["total_goals_team2"]
    matches["matches_diff"] = matches["total_matches_team1"] - matches["total_matches_team2"]
    matches["avg_goals_diff"] = matches["avg_goals_per_match_team1"] - matches["avg_goals_per_match_team2"]
    matches["num_players_with_goals_diff"] = matches["num_players_with_goals_team1"] - matches["num_players_with_goals_team2"]
    matches["max_goals_diff"] = matches["max_goals_by_single_player_team1"] - matches["max_goals_by_single_player_team2"]

    return matches


### function to predict winner of matches

In [31]:
def predict_matches(matches):
    X = matches[feature_cols]
    matches["pred_prob_win"] = clf.predict_proba(X)[:, 1]

    predictions = matches.loc[:, [
        "match_id",
        "team1",
        "team2",
        "date",
        "round",
        "pred_prob_win"
    ]].copy()

    predictions["predicted_winner"] = predictions.apply(
        lambda row: row["team1"] if row["pred_prob_win"] >= 0.5 else row["team2"],
        axis=1
    )

    predictions["confidence"] = predictions["pred_prob_win"].apply(
        lambda p: p if p >= 0.5 else 1 - p
    )

    return predictions

### functions as data preparation for the next round of predictions

In [32]:
winner_lookup = {}
loser_lookup = {}

def replace_winner(value):
    if isinstance(value, str) and value.startswith("WINNER_"):
        idx = value.split("_")[1] 
        match_id = f"M-2026-{idx}"
        return winner_lookup.get(match_id, value)
    return value

def replaces_winners_with_predictions(predictions):
    winner_lookup.update(
        predictions
        .set_index("match_id")["predicted_winner"]
        .to_dict()
    )

    df_matches["team1"] = df_matches["team1"].apply(replace_winner)
    df_matches["team2"] = df_matches["team2"].apply(replace_winner)


def replace_loser(value):
    if isinstance(value, str) and value.startswith("LOSER_"):
        idx = value.split("_")[1]  
        match_id = f"M-2026-{idx}"
        return loser_lookup.get(match_id, value)
    return value

def replaces_losers_with_predictions(predictions):
    predictions["predicted_loser"] = predictions.apply(
        lambda row: row["team1"] if row["pred_prob_win"] < 0.5 else row["team2"],
        axis=1
    )

    loser_lookup.update(
        predictions
        .set_index("match_id")["predicted_loser"]
        .to_dict()
    )

    df_matches["team1"] = df_matches["team1"].apply(replace_loser)
    df_matches["team2"] = df_matches["team2"].apply(replace_loser)

    predictions.drop(columns=["predicted_loser"], inplace=True)

## Orchestrating functions to predict world cup winner

In [33]:
matches_16_final = df_matches.iloc[0:16].copy()
matches_16_final = join_team_features_to_matches(matches_16_final)
predictions_16_final = predict_matches(matches_16_final)
replaces_winners_with_predictions(predictions_16_final)

matches_8_final = df_matches.iloc[16:24].copy()
matches_8_final = join_team_features_to_matches(matches_8_final)
predictions_8_final = predict_matches(matches_8_final)
replaces_winners_with_predictions(predictions_8_final)

matches_quarter_final = df_matches.iloc[24:28].copy()
matches_quarter_final = join_team_features_to_matches(matches_quarter_final)
predictions_quarter_final = predict_matches(matches_quarter_final)
replaces_winners_with_predictions(predictions_quarter_final)

matches_semi_final = df_matches.iloc[28:30].copy()
matches_semi_final = join_team_features_to_matches(matches_semi_final)
predictions_semi_final = predict_matches(matches_semi_final)
replaces_winners_with_predictions(predictions_semi_final)
replaces_losers_with_predictions(predictions_semi_final)

matches_final = df_matches.iloc[30:32].copy()
matches_final = join_team_features_to_matches(matches_final)
predictions_final = predict_matches(matches_final)
replaces_winners_with_predictions(predictions_final)

result = pd.concat([predictions_16_final, predictions_8_final, predictions_quarter_final, predictions_semi_final, predictions_final])

print(f"And the WINNER is {result.iloc[-1]['predicted_winner']} ⚽🏆🎉")

And the WINNER is Germany ⚽🏆🎉


## Output and visualization fo results

In [34]:
result.to_csv(
    "predictions/predictions_2026_knockout_matches_complete.csv",
    index=False
)

## Visualize tournament graph

In [35]:
emoji_lookup = {
    "Argentina": "🇦🇷",
    "Australia": "🇦🇺",
    "Austria": "🇦🇹",
    "Belgium": "🇧🇪",
    "Brazil": "🇧🇷",
    "Canada": "🇨🇦",
    "Cape Verde": "🇨🇻",
    "Colombia": "🇨🇴",
    "Croatia": "🇭🇷",
    "Ecuador": "🇪🇨",
    "Egypt": "🇪🇬",
    "England": "🏴󠁧󠁢󠁥󠁮󠁧󠁿 ",          
    "France": "🇫🇷",
    "Germany": "🇩🇪",
    "Haiti": "🇭🇹",
    "Iran": "🇮🇷",
    "Ivory Coast": "🇨🇮",
    "Japan": "🇯🇵",
    "Mexico": "🇲🇽",
    "Netherlands": "🇳🇱",
    "Portugal": "🇵🇹",
    "Qatar": "🇶🇦",
    "Scotland": "🏴󠁧󠁢󠁳󠁣󠁴󠁿 ",     
    "Scottland": "🏴󠁧󠁢󠁳󠁣󠁴󠁿 ",        
    "Senegal": "🇸🇳",
    "South Africa": "🇿🇦",
    "South Korea": "🇰🇷",
    "Spain": "🇪🇸",
    "United States": "🇺🇸",
    "Uruguay": "🇺🇾",
    "Czech Republic": "",
    "Bosnia and Herzegovina": "",
    "Turkey": "",
    "Sweden": "",
    "Iraq": "",
    "Kongo": "",
}

In [36]:
width = 15 
def print_match(idx, team1, team2, winner):
    if (idx % 2 == 0):
        prefix_quarter = "" if (((idx-2)%4) != 0) else f"{'':<{width*1.3}}  |"
        suffix_quarter = "" if (idx < 2 or (idx > 5 and idx < 10) or idx > 13) else "  |"
        line1 = f"{team1:<{width}} ─┐ {'':<{width*1.3}} {prefix_quarter}"
        line2 = f"{'':<{width}}  ├──▶ {winner:<{width}}─┐{prefix_quarter}"
        line3 = f"{team2:<{width}} ─┘{'':<{width*1.3}} |"
        print(f"{line1:<{80}}{suffix_quarter}")
        print(f"{line2:<{80}}{suffix_quarter}")
        print(f"{line3:<{80}}{suffix_quarter}")
    else: 
        prefix_quarter = "" if (((idx-1)%4) != 0) else f"{'':<{width*1.3}}  |"
        suffix_quarter = "" if (idx < 2 or (idx > 5 and idx < 10) or idx > 13) else "  |"
        line1 = f"{team1:<{width}} ─┐{'':<{width*1.3}} |"
        line2 = f"{'':<{width}}  ├──▶ {winner:<{width}}─┘{prefix_quarter}"
        line3 = f"{team2:<{width}} ─┘ {'':<{width*1.3}} {prefix_quarter}"
        print(f"{line1:<{80}}{suffix_quarter}")
        print(f"{line2:<{80}}{suffix_quarter}")
        print(f"{line3:<{80}}{suffix_quarter}")

def mapBosnia(team):
    if team == "Bosnia and Herzegovina":
        return "Bosnia"
    return team

def rearrange_match_order_for_visualization():
    swap_pairs = [(1, 2), (3, 4), (4, 10), (5, 11), (6, 8), (7, 9), (8, 10), (9, 11), (12, 13), (13, 15), (14, 15)]
    for i, j in swap_pairs:
        predictions_16_final.iloc[[i, j]] = predictions_16_final.iloc[[j, i]].values

    predictions_8_final.iloc[[2, 4]] = predictions_8_final.iloc[[4, 2]].values
    predictions_8_final.iloc[[3, 5]] = predictions_8_final.iloc[[5, 3]].values

predictions_16_final = predictions_16_final.sort_values("match_id")
predictions_8_final = predictions_8_final.sort_values("match_id")
rearrange_match_order_for_visualization()
for idx, r in predictions_16_final.iterrows():
    print_match(
        idx,
        mapBosnia(r["team1"]),
        mapBosnia(r["team2"]),
        mapBosnia(r["predicted_winner"])
    )
    
    if (idx % 2 == 0):
        winner_8_idx = int(idx / 2)
        edge = "─┐" if ((winner_8_idx % 2) == 0) else "─┘"
        winner = f"{predictions_8_final.iloc[winner_8_idx]['predicted_winner']}"
        final_line = "" if (idx < 4 or idx > 10) else f"{'':<{45}}|"
        print(f"{'':<{width*2.5}} ├──▶  {winner:<{width}}{edge}{final_line}") 
    else:
        if ((idx-1) % 4 == 0): # quarter final
            winner_4_idx = int((idx-1) / 4)
            edge = "─┐" if ((winner_4_idx % 2) == 0) else "─┘"
            winner = f"{predictions_quarter_final.iloc[winner_4_idx]['predicted_winner']}"
            final_line = "" if (idx < 4 or idx > 10) else f"{'':<{23}}|"
            print(f"{'':<{width*4}}├──▶  {winner:<{width}}{edge}{final_line}") 
        else: 
            if (idx == 3 or idx == 11): # semi final
                winner_2_idx = 0 if (idx == 3) else 1
                edge = "─┐" if (winner_2_idx == 0) else "─┘"
                winner = f"{predictions_semi_final.iloc[winner_2_idx]['predicted_winner']}"
                print(f"{'':<{width*5.5}}├──▶  {winner:<{width}}  {edge}") 
            else:
                if (idx == 7): # final
                    winner = f"{predictions_final.iloc[1]['predicted_winner']}"
                    winner_line = f"{winner} {emoji_lookup.get(winner, '')}"
                    print(f"{'':<{100}} 🏆 {winner_line:<{width}}") 



South Africa    ─┐                                                              
                 ├──▶ South Africa   ─┐                                         
Switzerland     ─┘                    |                                         
                                      ├──▶  Netherlands    ─┐
Netherlands     ─┐                    |                                         
                 ├──▶ Netherlands    ─┘                     |                   
Scotland        ─┘                                          |                   
                                                            ├──▶  Germany        ─┐
Germany         ─┐                                          |                     |
                 ├──▶ Germany        ─┐                     |                     |
Czech Republic  ─┘                    |                                           |
                                      ├──▶  Germany        ─┘
Senegal         ─┐                    |               